# Panoramic X-ray to 3D scan: scoring the trained model (Kaggle)

This notebook does not train anything. It takes about 5 minutes.

1. It loads the trained model (`best.pt`) from the output of the notebook `xray-to-3d-training`.
2. It runs the model on the 5 check scans. Their scores must be the same as the scores the training printed. This shows the model file was loaded correctly.
3. It runs the model on the 10 test scans. These scans were never used during training.
4. It measures how far the model's output is from the real 3D scan, in millimeters.

**To see the results:** open the "Output" tab. `results/test/results.csv` holds the numbers for each scan,
`results/test/overview_NUMBER.png` and `results/test/cross_NUMBER.png` are pictures, and
`predictions/test/NUMBER.npz` is the model's output for each scan.

In [ ]:
import torch
assert torch.cuda.is_available(), "No graphics card. In the right panel choose Session options > Accelerator > GPU T4 x2, then run again."
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
import glob, os, shutil
files = sorted(glob.glob("/kaggle/input/**/*.npz", recursive=True))
assert files, "No prepared scans were found. Click 'Add Input' and add the dataset xray-prepared-scans."
os.makedirs("/tmp/prepared", exist_ok=True)
for f in files:
    shutil.copy(f, "/tmp/prepared/")
print(len(files), "prepared scans copied")

models = sorted(glob.glob("/kaggle/input/**/runs/full/best.pt", recursive=True))
assert models, "The trained model was not found. Click 'Add Input' and add the output of the notebook xray-to-3d-training."
MODEL = models[-1]
print("model file:", MODEL, round(os.path.getsize(MODEL) / 1e6), "MB")

In [ ]:
os.makedirs("/kaggle/working/project/src", exist_ok=True)
os.makedirs("/kaggle/working/project/scripts", exist_ok=True)

The next 6 cells write the project's code files into Kaggle. They are copies of the files in the project folder.

In [ ]:
%%writefile /kaggle/working/project/src/fake_panoramic.py
"""Make a fake panoramic X-ray from a 3D CBCT.

How it works, in plain steps:
1. Find the curve of the jaw, seen from above (the U shape), at the height of the tooth roots.
2. Walk along that curve. At each point, look straight across the curve, through the jaw.
3. Add up how much X-ray each bit of tissue along that short line would block. Bone blocks a lot, air almost none.
   Tissue near the curve counts fully; tissue further away counts less (real machines blur it out).
4. Each point on the curve becomes one column of the picture; each height in the scan becomes one row.

Volumes are indexed [z, y, x]. z grows toward the top of the head, y grows toward the back of the head.
"""
import numpy as np
from scipy.ndimage import binary_opening, gaussian_filter1d, label, map_coordinates, zoom

MU_WATER = 0.02  # how much 1 mm of water blocks X-rays (per mm), a typical value near 70 keV


def hu_to_mu(vol_hu: np.ndarray, metal_cap: float = 3000.0) -> np.ndarray:
    """Turn CT numbers (Hounsfield units) into how much each mm of tissue blocks X-rays."""
    hu = np.clip(vol_hu, -1000, metal_cap)
    return MU_WATER * (1 + hu / 1000.0)


def resample(vol: np.ndarray, spacing, new_spacing: float) -> np.ndarray:
    return zoom(vol, [s / new_spacing for s in spacing], order=1)


def find_bite_level(vol_hu: np.ndarray, teeth_hu: float = 1800) -> int:
    """Slice index where the most very bright voxels (tooth enamel, fillings) are: roughly where the teeth meet."""
    counts = (vol_hu > teeth_hu).sum(axis=(1, 2))
    return int(np.argmax(gaussian_filter1d(counts.astype(float), 3)))


def find_arch_curve(vol_hu: np.ndarray, z_lo: int, z_hi: int, bone_hu: float = 500, metal_hu: float = 2800,
                    angle_range_deg: float = 100, n_angles: int = 201):
    """Find the jaw's U shape, seen from above, between slices z_lo and z_hi.

    Keeps bone but not metal, keeps the biggest connected piece (the jaw), then stands at a point
    inside the U and looks outward in many directions. Along each direction, the middle of the
    last stretch of bone before the outside of the face is one point of the curve.
    The distances are smoothed so the curve has no jumps.
    Returns (points_x, points_y, center).
    """
    slab = vol_hu[z_lo:z_hi]
    bone = ((slab > bone_hu) & (slab < metal_hu)).any(axis=0)
    bone = binary_opening(bone, iterations=2)
    lab, _ = label(bone)
    sizes = np.bincount(lab.ravel()); sizes[0] = 0
    bone = lab == sizes.argmax()
    return arch_curve_from_shape(bone, angle_range_deg, n_angles)


def arch_curve_from_shape(bone: np.ndarray, angle_range_deg: float = 100, n_angles: int = 201):
    """The second half of find_arch_curve. `bone` is a yes/no picture of the jaw seen from above."""
    ys, xs = np.nonzero(bone)
    cx = (xs.min() + xs.max()) / 2
    cy = ys.max() - 0.15 * (ys.max() - ys.min())  # inside the U, near the back ends of the jaw
    angles = np.deg2rad(np.linspace(-angle_range_deg, angle_range_deg, n_angles))  # 0 = straight to the front
    r_max = np.hypot(*bone.shape)
    radii = np.arange(0, r_max, 0.5)
    r_mid = np.full(len(angles), np.nan)
    for i, a in enumerate(angles):
        dx, dy = np.sin(a), -np.cos(a)
        px, py = cx + radii * dx, cy + radii * dy
        inside = (px >= 0) & (px < bone.shape[1] - 1) & (py >= 0) & (py < bone.shape[0] - 1)
        hit = np.zeros(len(radii), bool)
        hit[inside] = bone[np.round(py[inside]).astype(int), np.round(px[inside]).astype(int)]
        idx = np.flatnonzero(hit)
        if idx.size == 0:
            continue
        end = idx[-1]
        start = end
        while start - 1 >= 0 and hit[start - 1]:
            start -= 1
        r_mid[i] = radii[(start + end) // 2]
    ok = ~np.isnan(r_mid)
    # smooth the distance-vs-direction curve with a low-order polynomial, dropping outliers
    keep = ok.copy()
    for _ in range(3):
        coef = np.polyfit(angles[keep], r_mid[keep], 6)
        res = np.abs(r_mid - np.polyval(coef, angles))
        keep = ok & (res < max(3 * np.nanmedian(res[keep]), 2.0))
    a_used = angles[ok]
    a_fine = np.linspace(a_used.min(), a_used.max(), 2000)
    r_fine = np.polyval(coef, a_fine)
    return cx + r_fine * np.sin(a_fine), cy - r_fine * np.cos(a_fine), (cx, cy), (angles[keep], r_mid[keep])


def curve_points(cx_pts, cy_pts, center, step: float):
    """Points spaced `step` voxels apart along the curve, plus the direction across the curve (pointing outward)."""
    seg = np.hypot(np.diff(cx_pts), np.diff(cy_pts))
    arc = np.concatenate([[0], np.cumsum(seg)])
    s = np.arange(0, arc[-1], step)
    px, py = np.interp(s, arc, cx_pts), np.interp(s, arc, cy_pts)
    tx, ty = np.gradient(px), np.gradient(py)
    nx, ny = ty, -tx
    norm = np.hypot(nx, ny)
    nx, ny = nx / norm, ny / norm
    flip = (px - center[0]) * nx + (py - center[1]) * ny < 0
    nx[flip] *= -1; ny[flip] *= -1
    return px, py, nx, ny


def make_panoramic(mu: np.ndarray, curve_x, curve_y, center, voxel_mm: float,
                   depth_in_mm: float = 150, depth_out_mm: float = 30, sharp_mm: float = 4,
                   tilt_deg: float = 7, near_mm: float = 25, near_blur_px: float = 6,
                   far_blur_px: float = 25) -> np.ndarray:
    """Return the fake panoramic picture: rows = height (top of head first), columns = along the jaw
    (patient's right side on the left of the picture, like a real panoramic X-ray).

    Each column's light line starts outside the face and goes depth_in_mm into the head, so it
    passes through the whole head (tongue, throat, spine), as in a real machine.
    The line tilts upward by tilt_deg toward the front, like a real machine's beam.
    Everything along the line blocks light, but only tissue within about sharp_mm of the jaw curve is
    sharp. Tissue up to near_mm away is smeared sideways by near_blur_px columns, and tissue further
    away (for example the spine) by far_blur_px, because the machine moves while taking the picture.
    """
    px, py, nx, ny = curve_points(curve_x, curve_y, center, step=1.0)
    t = np.arange(-depth_in_mm, depth_out_mm, voxel_mm) / voxel_mm  # negative = into the head
    d = np.abs(t * voxel_mm)
    w_sharp = np.exp(-0.5 * (d / sharp_mm) ** 2)
    w_near = (1 - w_sharp) * (d <= near_mm)
    w_far = (1 - w_sharp) * (d > near_mm)
    dz = t * np.tan(np.deg2rad(tilt_deg))  # points deeper in the head sit lower on the line
    nz = mu.shape[0]
    sharp, near, far = (np.zeros((nz, len(px)), np.float32) for _ in range(3))
    zz = np.arange(nz)[:, None] + dz[None, :]
    for i in range(len(px)):
        sx = np.broadcast_to(px[i] + t * nx[i], zz.shape)
        sy = np.broadcast_to(py[i] + t * ny[i], zz.shape)
        vals = map_coordinates(mu, np.stack([zz.ravel(), sy.ravel(), sx.ravel()]), order=1, cval=0.0)
        vals = vals.reshape(zz.shape) * voxel_mm
        sharp[:, i] = vals @ w_sharp
        near[:, i] = vals @ w_near
        far[:, i] = vals @ w_far
    img = sharp + gaussian_filter1d(near, near_blur_px, axis=1) + gaussian_filter1d(far, far_blur_px, axis=1)
    return img[::-1]


def look_like_real(img: np.ndarray, clip_limit: float = 1.2, sharpen: float = 0.6) -> np.ndarray:
    """Imitate the picture clean-up that panoramic machines do: gentle local brightness balancing
    and edge sharpening. Returns values between 0 and 1."""
    import cv2
    lo, hi = np.percentile(img, [0.5, 99.8])
    x = np.clip((img - lo) / (hi - lo), 0, 1)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(4, 8))
    x = clahe.apply((x * 65535).astype(np.uint16)).astype(np.float32) / 65535
    blurred = cv2.GaussianBlur(x, (0, 0), 3)
    return np.clip(x + sharpen * (x - blurred), 0, 1)


In [ ]:
%%writefile /kaggle/working/project/src/curved_volume.py
"""Turn a 3D scan into a "straightened" 3D block that lines up with the fake panoramic X-ray.

The fake X-ray has one column for each point along the U-shaped jaw curve and one row for each height.
The straightened block has the same rows and columns, plus a third direction: how far a point is from
the jaw curve, measured straight across the curve (toward the tongue or toward the cheek).

So pixel (row, column) of the X-ray and the line block[row, column, :] show the same place in the head.
The model gets the X-ray as input and must output the block.

Arrays are indexed [z, y, x] for scans and [row, column, depth] for blocks. Row 0 is the top of the head.
"""
import numpy as np
from scipy.ndimage import map_coordinates

from .fake_panoramic import arch_curve_from_shape, curve_points, find_arch_curve, find_bite_level

VOXEL_MM = 0.4      # size of one voxel in the scans, and of one row or column in the X-ray and block
DEPTH_IN_MM = 22.0  # how far the block reaches from the jaw curve toward the tongue
DEPTH_OUT_MM = 10.0 # how far the block reaches from the jaw curve toward the cheek
DEPTH_STEP_MM = 0.5 # distance between two neighbouring depth samples
OUTSIDE = 255       # label value for places that are outside the scan


def depth_positions_mm() -> np.ndarray:
    """Distance from the jaw curve of each depth sample. Negative = toward the tongue."""
    return np.arange(-DEPTH_IN_MM, DEPTH_OUT_MM, DEPTH_STEP_MM)


def jaw_curve(vol_hu: np.ndarray):
    """Find the jaw curve of a scan. Returns (bite slice, a dict with the curve)."""
    bite = find_bite_level(vol_hu)
    z_lo, z_hi = max(bite - int(20 / VOXEL_MM), 0), bite + int(5 / VOXEL_MM)
    cx, cy, center, _ = find_arch_curve(vol_hu, z_lo, z_hi)
    px, py, nx, ny = curve_points(cx, cy, center, step=1.0)
    middle = int(np.argmin(np.abs(px - center[0]) + 1e3 * (py > center[1])))  # the front-most point of the U
    return bite, dict(cx=cx, cy=cy, center=np.array(center), px=px, py=py, nx=nx, ny=ny, middle=middle)


def straighten(vol: np.ndarray, curve: dict, is_label: bool):
    """Sample the scan (or label file) along the jaw curve. Returns block[row, column, depth].

    For a scan, places outside the scan get -1000 (air). For labels they get OUTSIDE.
    """
    t = depth_positions_mm() / VOXEL_MM
    sx = curve["px"][:, None] + t[None, :] * curve["nx"][:, None]
    sy = curve["py"][:, None] + t[None, :] * curve["ny"][:, None]
    coords = np.stack([sy, sx])
    out = np.empty((vol.shape[0], sx.shape[0], sx.shape[1]), np.uint8 if is_label else np.float32)
    for z in range(vol.shape[0]):
        if is_label:
            out[z] = map_coordinates(vol[z], coords, order=0, cval=OUTSIDE)
        else:
            out[z] = map_coordinates(vol[z], coords, order=1, cval=-1000.0)
    return out[::-1]


def bend_back(block: np.ndarray, curve: dict, shape_zyx, fill: float, is_label: bool = False) -> np.ndarray:
    """The opposite of straighten: put a block back into the normal [z, y, x] shape of a scan.

    For each (y, x) place it finds the nearest point on the jaw curve and the distance across the curve,
    then reads the block there. Places further from the curve than the block reaches get `fill`.
    """
    from scipy.spatial import cKDTree
    nz, ny_, nx_ = shape_zyx
    yy, xx = np.mgrid[0:ny_, 0:nx_].astype(np.float32)
    pts = np.stack([curve["px"], curve["py"]], axis=1)
    _, idx = cKDTree(pts).query(np.stack([xx.ravel(), yy.ravel()], axis=1))
    dx, dy = xx.ravel() - curve["px"][idx], yy.ravel() - curve["py"][idx]
    # distance across the curve (signed), and distance along the curve from the nearest curve point
    across = dx * curve["nx"][idx] + dy * curve["ny"][idx]
    along = dx * np.gradient(curve["px"])[idx] + dy * np.gradient(curve["py"])[idx]
    col = idx + along  # curve points are 1 voxel apart, so 1 voxel along the curve = 1 column
    dep = (across * VOXEL_MM + DEPTH_IN_MM) / DEPTH_STEP_MM
    n_col, n_dep = block.shape[1], block.shape[2]
    ok = (col >= 0) & (col <= n_col - 1) & (dep >= 0) & (dep <= n_dep - 1) & (np.abs(along) < 3)
    out = np.full((nz, ny_ * nx_), fill, block.dtype)
    coords = np.stack([col[ok], dep[ok]])
    for z in range(nz):
        out[z, ok] = map_coordinates(block[nz - 1 - z], coords, order=0 if is_label else 1, mode="nearest")
    return out.reshape(nz, ny_, nx_)


# Every X-ray and block is put on a frame of one fixed size, so the model always gets the same size.
# The row where the teeth meet and the column of the front of the jaw are always at the same place.
FRAME_ROWS = 240
FRAME_COLS = 544
FRAME_BITE_ROW = 80
FRAME_MIDDLE_COL = FRAME_COLS // 2


def to_frame(arr: np.ndarray, bite_row: int, middle_col: int, fill):
    """Copy arr[row, column, ...] onto the fixed frame. Parts that do not fit are dropped.
    Returns (frame, row shift, column shift), where frame[r + row shift, c + column shift] = arr[r, c]."""
    dr, dc = FRAME_BITE_ROW - bite_row, FRAME_MIDDLE_COL - middle_col
    out = np.full((FRAME_ROWS, FRAME_COLS) + arr.shape[2:], fill, arr.dtype)
    r0, r1 = max(0, -dr), min(arr.shape[0], FRAME_ROWS - dr)
    c0, c1 = max(0, -dc), min(arr.shape[1], FRAME_COLS - dc)
    out[r0 + dr:r1 + dr, c0 + dc:c1 + dc] = arr[r0:r1, c0:c1]
    return out, dr, dc


def jaw_curve_from_labels(lab: np.ndarray):
    """Find the bite slice and the jaw curve of a training scan from its label file
    (the original ToothFairy3 numbers, not the merged ones).

    This is more reliable than jaw_curve, which has to guess from brightness alone.
    Bite slice = halfway between the tips of the lower teeth and the tips of the upper teeth.
    Jaw curve = the U shape of the lower jaw bone and the teeth, seen from above, from 30 mm below
    the bite slice to 5 mm above it.
    """
    lower = ((lab >= 31) & (lab <= 48)) | ((lab >= 131) & (lab <= 148))
    upper = ((lab >= 11) & (lab <= 28)) | ((lab >= 111) & (lab <= 128))
    tips = []
    for teeth, q in ((lower, 97), (upper, 3)):
        per_slice = teeth.sum(axis=(1, 2))
        if per_slice.sum() > 0:
            tips.append(np.searchsorted(np.cumsum(per_slice) / per_slice.sum(), q / 100))
    if not tips:
        raise ValueError("no teeth in the label file")
    bite = int(round(np.mean(tips)))
    z_lo, z_hi = max(bite - int(30 / VOXEL_MM), 0), min(bite + int(5 / VOXEL_MM), lab.shape[0])
    slab = lab[z_lo:z_hi]
    shape = ((slab == 1) | ((slab >= 8) & (slab <= 48)) | ((slab >= 111) & (slab <= 148))).any(axis=0)
    return bite, curve_from_shape(shape)


def curve_from_shape(shape: np.ndarray) -> dict:
    """The jaw curve from a yes/no picture of the lower jaw bone and the teeth, seen from above."""
    cx, cy, center, _ = arch_curve_from_shape(shape)
    # keep only the part of the curve that goes outward from the front; drop ends that turn back
    angle = np.arctan2(cx - center[0], center[1] - cy)
    radius = np.hypot(cx - center[0], cy - center[1])
    front = int(np.argmin(np.abs(angle) + 10 * (radius < 0.5 * np.median(radius))))
    good = radius > 0.35 * radius[front]
    # also drop ends that turn back toward the front, and ends that are past the last bit of jaw
    for side in (slice(front, None), slice(front, None, -1)):
        y = cy[side]
        good[side] &= np.cumsum(y < np.maximum.accumulate(y) - 1.0) == 0
    from scipy.ndimage import binary_dilation
    near_jaw = binary_dilation(shape, iterations=3)
    on_jaw = np.flatnonzero(near_jaw[np.clip(np.round(cy).astype(int), 0, shape.shape[0] - 1),
                                     np.clip(np.round(cx).astype(int), 0, shape.shape[1] - 1)])
    good[:on_jaw[0]] = False
    good[on_jaw[-1] + 1:] = False
    first = front - np.argmax(~good[front::-1]) + 1 if (~good[:front]).any() else 0
    last = front + np.argmax(~good[front:]) if (~good[front:]).any() else len(cx)
    cx, cy = cx[first:last], cy[first:last]
    px, py, nx, ny = curve_points(cx, cy, center, step=1.0)
    middle = int(np.argmin(np.hypot(px - cx[front - first], py - cy[front - first])))
    return dict(cx=cx, cy=cy, center=np.array(center), px=px, py=py, nx=nx, ny=ny, middle=middle)


def jaw_curve_from_brightness(vol_hu: np.ndarray, bite: int, hard_hu: float = 900):
    """The jaw curve of a scan that has no label file, found the same way as for the training scans:
    everything hard (bone, teeth, fillings) from 30 mm below the bite slice to 5 mm above it, seen
    from above, is used in place of the labels. The biggest connected piece is the jaw."""
    from scipy.ndimage import binary_fill_holes, binary_opening, label
    z_lo, z_hi = max(bite - int(30 / VOXEL_MM), 0), min(bite + int(5 / VOXEL_MM), vol_hu.shape[0])
    shape = binary_opening((vol_hu[z_lo:z_hi] > hard_hu).any(axis=0), iterations=2)
    pieces, _ = label(shape)
    sizes = np.bincount(pieces.ravel()); sizes[0] = 0
    return curve_from_shape(binary_fill_holes(pieces == sizes.argmax()))


In [ ]:
%%writefile /kaggle/working/project/src/model.py
"""The model: gets a panoramic X-ray, outputs the straightened 3D block.

Plain description:
1. A "U-Net" looks at the X-ray. A U-Net is a common program for pictures. It shrinks the picture
   step by step to find large shapes (a whole jaw), then grows it back step by step to find small
   shapes (one tooth root), and at each size it reuses what it found on the way down.
2. For each X-ray pixel the U-Net outputs a list of numbers. The list is turned into a short line
   of points going from the tongue side to the cheek side. All lines together are the 3D block.
3. A small 3D step looks at each point together with its neighbours in all three directions and
   outputs, for each point: how bright it is in a 3D scan, and which part it is (bone, tooth, ...).
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


def conv_block(n_in, n_out):
    return nn.Sequential(
        nn.Conv2d(n_in, n_out, 3, padding=1), nn.GroupNorm(8, n_out), nn.ReLU(inplace=True),
        nn.Conv2d(n_out, n_out, 3, padding=1), nn.GroupNorm(8, n_out), nn.ReLU(inplace=True))


class XrayTo3D(nn.Module):
    def __init__(self, in_channels=1, depth=64, n_classes=6, widths=(48, 96, 192, 384, 512), lift=8):
        """in_channels: 1 for the X-ray alone. More when extra inputs (from the mouth video) are added.
        depth: number of points on each tongue-to-cheek line."""
        super().__init__()
        self.depth, self.n_classes, self.lift = depth, n_classes, lift
        self.down = nn.ModuleList()
        n = in_channels
        for w in widths:
            self.down.append(conv_block(n, w))
            n = w
        self.up = nn.ModuleList()
        for w in reversed(widths[:-1]):
            self.up.append(conv_block(n + w, w))
            n = w
        self.to_lines = nn.Conv2d(n, lift * depth, 1)
        self.head3d = nn.Sequential(
            nn.Conv3d(lift, 24, 3, padding=1), nn.GroupNorm(8, 24), nn.ReLU(inplace=True),
            nn.Conv3d(24, 24, 3, padding=1), nn.GroupNorm(8, 24), nn.ReLU(inplace=True),
            nn.Conv3d(24, 1 + n_classes, 1))

    def forward(self, xray):
        """xray: [batch, in_channels, rows, columns], rows and columns must be multiples of 16.
        Returns (brightness [batch, depth, rows, columns] between 0 and 1,
                 part scores [batch, n_classes, depth, rows, columns])."""
        kept = []
        x = xray
        for i, block in enumerate(self.down):
            x = block(x)
            if i < len(self.down) - 1:
                kept.append(x)
                x = F.max_pool2d(x, 2)
        for block in self.up:
            x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
            x = block(torch.cat([x, kept.pop()], dim=1))
        b, _, h, w = x.shape
        x = self.to_lines(x).view(b, self.lift, self.depth, h, w)
        x = self.head3d(x)
        return torch.sigmoid(x[:, 0]), x[:, 1:]


In [ ]:
%%writefile /kaggle/working/project/src/train_data.py
"""Load the prepared examples and cut random training pieces from them."""
import json
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

OUTSIDE = 255
HU_LOW, HU_HIGH = -1000.0, 3000.0


def hu_to_unit(hu):
    """Hounsfield units -> numbers between 0 and 1 (what the model outputs)."""
    return (hu - HU_LOW) / (HU_HIGH - HU_LOW)


def unit_to_hu(x):
    return x * (HU_HIGH - HU_LOW) + HU_LOW


def make_split(folder, n_check=5, n_test=10, seed=0):
    """Split the scans into train / check / test. Saved in split.json so it never changes.
    "check" scans are used during training to see if the model is getting better.
    "test" scans are never used during training."""
    folder = Path(folder)
    path = folder / "split.json"
    if path.exists():
        return json.loads(path.read_text())
    nums = sorted(p.stem for p in folder.glob("*.npz"))
    random.Random(seed).shuffle(nums)
    split = dict(test=sorted(nums[:n_test]), check=sorted(nums[n_test:n_test + n_check]),
                 train=sorted(nums[n_test + n_check:]))
    path.write_text(json.dumps(split, indent=1))
    return split


class Examples:
    """Holds a list of prepared scans in memory.
    xray: [n, rows, columns] 0..1,  hu: [n, depth, rows, columns] Hounsfield units,
    labels: [n, depth, rows, columns] part numbers, 255 = outside the scan."""

    def __init__(self, folder, numbers):
        xs, hs, ls = [], [], []
        for num in numbers:
            with np.load(Path(folder) / f"{num}.npz") as f:
                xs.append(torch.from_numpy(f["xray"].astype(np.float32)))
                hs.append(torch.from_numpy(f["hu"]).permute(2, 0, 1).contiguous())
                ls.append(torch.from_numpy(f["labels"]).permute(2, 0, 1).contiguous())
        self.numbers = list(numbers)
        self.xray, self.hu, self.labels = torch.stack(xs), torch.stack(hs), torch.stack(ls)

    def __len__(self):
        return len(self.numbers)

    def whole(self, i):
        """One whole example, as a batch of 1."""
        return self.xray[i:i + 1, None], self.hu[i:i + 1], self.labels[i:i + 1]

    def random_batch(self, batch_size, piece_cols, max_shift_rows=8):
        """Random pieces for training. Each piece is piece_cols columns wide and is randomly
        mirrored left-right and moved up or down a little, so the model sees more variety."""
        rows, cols = self.xray.shape[1:]
        xs, hs, ls = [], [], []
        for _ in range(batch_size):
            i = random.randrange(len(self))
            c = random.randrange(0, cols - piece_cols + 1)
            x, h, l = self.xray[i, :, c:c + piece_cols], self.hu[i, :, :, c:c + piece_cols], self.labels[i, :, :, c:c + piece_cols]
            if random.random() < 0.5:
                x, h, l = x.flip(-1), h.flip(-1), l.flip(-1)
            s = random.randint(-max_shift_rows, max_shift_rows)
            if s != 0:
                x, h, l = _shift_rows(x, s, 0.0), _shift_rows(h, s, int(HU_LOW)), _shift_rows(l, s, OUTSIDE)
            xs.append(x); hs.append(h); ls.append(l)
        return torch.stack(xs)[:, None], torch.stack(hs), torch.stack(ls)


def _shift_rows(t, s, fill):
    """Move the picture s rows down (or up if s is negative). Rows that become empty get `fill`."""
    out = torch.full_like(t, fill)
    if s > 0:
        out[..., s:, :] = t[..., :-s, :]
    else:
        out[..., :s, :] = t[..., -s:, :]
    return out


def change_xray_look(xray):
    """Randomly change brightness curve, contrast, blur and noise of a batch of X-rays [b, 1, rows, columns].
    Real X-rays do not look exactly like our fake ones, so the model must not depend on the exact look."""
    b = xray.shape[0]
    dev = xray.device
    gamma = torch.empty(b, 1, 1, 1, device=dev).uniform_(0.7, 1.5)
    x = xray.clamp(0, 1) ** gamma
    contrast = torch.empty(b, 1, 1, 1, device=dev).uniform_(0.75, 1.25)
    bright = torch.empty(b, 1, 1, 1, device=dev).uniform_(-0.08, 0.08)
    x = (x - 0.5) * contrast + 0.5 + bright
    if random.random() < 0.5:
        sigma = random.uniform(0.5, 2.0)
        k = torch.arange(-6, 7, device=dev, dtype=x.dtype)
        k = torch.exp(-0.5 * (k / sigma) ** 2)
        k = k / k.sum()
        x = F.conv2d(F.pad(x, (6, 6, 0, 0), mode="replicate"), k.view(1, 1, 1, -1))
        x = F.conv2d(F.pad(x, (0, 0, 6, 6), mode="replicate"), k.view(1, 1, -1, 1))
    x = x + torch.randn_like(x) * random.uniform(0.0, 0.04)
    return x.clamp(0, 1)


In [ ]:
%%writefile /kaggle/working/project/scripts/predict.py
"""Run the trained model on prepared scans and save what it outputs.

Run:  python scripts/predict.py --data PREPARED_FOLDER --model best.pt --out predictions/test --group test

For each scan this gives the model the fake X-ray of that scan and saves the model's output in one
file, SCAN_NUMBER.npz, with:
  hu      the brightness of every point of the block, in Hounsfield units, [rows, columns, depth]
  labels  the part number of every point of the block (0 background, 1 lower jaw, 2 upper jaw,
          3 nerve canal, 4 sinus, 5 teeth), [rows, columns, depth]
This is the same layout as the prepared scan files, so the two can be compared point by point.
The comparing is done by scripts/measure.py.

--group says which scans to use:
  test   the 10 scans that were never used during training (the default)
  check  the 5 scans that were used during training to pick the best saved model
  train  the 46 scans the model was trained on
"""
import argparse
import json
import sys
import time
from pathlib import Path

import numpy as np
import torch

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))
from src.model import XrayTo3D
from src.train_data import make_split, unit_to_hu


def load_model(path, device):
    """Load a model file saved by scripts/train.py."""
    saved = torch.load(path, map_location=device, weights_only=False)  # our own file, so it is safe to load fully
    widths = (16, 32, 48, 64, 64) if saved.get("small") else (48, 96, 192, 384, 512)
    model = XrayTo3D(depth=saved["depth"], widths=widths).to(device)
    model.load_state_dict(saved["model"])
    model.eval()
    return model, saved


@torch.no_grad()
def predict_one(model, xray, device):
    """xray: [rows, columns] numbers from 0 to 1. Returns (hu, labels), both [rows, columns, depth]."""
    x = torch.from_numpy(xray.astype(np.float32))[None, None].to(device)
    with torch.autocast(device.type, enabled=device.type == "cuda"):
        bright, scores = model(x)
    hu = unit_to_hu(bright[0].float()).round().clamp(-1000, 3000).to(torch.int16)
    labels = scores[0].argmax(dim=0).to(torch.uint8)
    return hu.permute(1, 2, 0).cpu().numpy(), labels.permute(1, 2, 0).cpu().numpy()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True, help="folder made by prepare_tf3.py")
    ap.add_argument("--model", required=True, help="model file saved by train.py, for example best.pt")
    ap.add_argument("--out", required=True, help="folder for the model's outputs")
    ap.add_argument("--group", default="test", choices=["test", "check", "train"])
    ap.add_argument("--scans", nargs="*", help="scan numbers to use instead of a whole group")
    args = ap.parse_args()

    out = Path(args.out); out.mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"running on: {device}" + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""), flush=True)

    numbers = args.scans if args.scans else make_split(args.data)[args.group]

    # The training script saved the list of scans it trained on next to the model file.
    # Stop if a scan we are about to score as "never used" was in fact used.
    settings_path = Path(args.model).parent / "settings.json"
    if settings_path.exists():
        settings = json.loads(settings_path.read_text())
        used = set(settings["train"]) | (set(settings["check"]) if args.group == "test" else set())
        if args.group != "train":
            overlap = sorted(set(numbers) & used)
            if overlap:
                raise SystemExit(f"ERROR: these scans were used during training: {overlap}")
            print(f"checked against {settings_path.name}: none of the {len(numbers)} scans was used for training", flush=True)
    else:
        print("no settings.json next to the model file, so the scan list could not be checked", flush=True)

    model, saved = load_model(args.model, device)
    print(f"loaded {args.model} (saved after epoch {saved['epoch']})", flush=True)
    for i, num in enumerate(numbers):
        t0 = time.time()
        with np.load(Path(args.data) / f"{num}.npz") as f:
            xray = f["xray"]
        hu, labels = predict_one(model, xray, device)
        np.savez_compressed(out / f"{num}.npz", hu=hu, labels=labels)
        print(f"[{i + 1}/{len(numbers)}] scan {num} done in {time.time() - t0:.1f}s", flush=True)
    (out / "about.json").write_text(json.dumps(dict(model=str(args.model), epoch=saved["epoch"], group="given list" if args.scans else args.group, scans=list(numbers)), indent=1))
    print(f"{len(numbers)} outputs are in {out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/project/scripts/measure.py
"""Compare the model's outputs with the correct answers and measure the difference in millimeters.

Run:  python scripts/measure.py --data PREPARED_FOLDER --pred predictions/test --out results/test

--data  folder made by prepare_tf3.py (holds the correct answer for every scan)
--pred  folder made by predict.py (holds the model's output for some scans)
--out   folder for the numbers (results.csv, summary.json) and the pictures

Everything is measured in the block. In the block one row is 0.4 mm of height, one column is 0.4 mm
along the jaw curve, and one depth step is 0.5 mm from the tongue side toward the cheek side.

What is measured for each scan:

1. Nerve canal (the tube in the lower jaw bone that holds the nerve).
   For each column that holds canal, the middle point of the canal in that column is found. All
   middle points together are the "middle line" of the canal. Then:
     "canal mm"          average distance between the model's middle line and the true middle line.
                         It is the average of two numbers: from each true point to the nearest model
                         point, and from each model point to the nearest true point.
     "canal found share" share of the true middle line that has a model point within 2 mm
                         (1 = the model found the whole canal, 0 = it found nothing).

2. Sinus (the air space above the upper back teeth). Only its bottom edge is in the training scans.
   For each column that holds sinus, the lowest row with sinus is the bottom edge.
     "sinus edge mm"        average height difference between the model's edge and the true edge.
     "sinus edge signed mm" the same, but with a sign: positive = the model put the edge too low.
     "sinus columns found share" share of the true sinus columns in which the model output any sinus.

3. Lower jaw thickness, measured from the tongue side to the cheek side, at two heights
   (12 mm and 20 mm below the line where the teeth meet), at every column within 45 mm of the
   front of the jaw (the part of the jaw that holds teeth). Tooth roots and the nerve canal count
   as part of the jaw.
     "jaw thickness true mm 12", "jaw thickness model mm 12"  average thickness
     "jaw thickness diff mm 12"   average difference per column (always counted as positive)

4. Bone density. Brightness in a 3D scan is given in Hounsfield units (air is -1000, water is 0,
   dense bone is above 1000). Brighter means denser.
     "lower jaw HU true", "lower jaw HU model"  average brightness inside the true lower jaw bone
     "upper jaw HU true", "upper jaw HU model"  the same for the upper jaw bone
     "bone err HU"  average brightness difference, point by point, inside the true bone

5. The same numbers the training script printed: "err_hu" (average brightness difference over the
   whole block), "psnr" (a standard score for how close two pictures are, higher is better), and
   "dice PART" (overlap score from 0 to 1: 1 means the model put the part in exactly the right place).

Before the distances for the canal and the sinus are measured, pieces of the model's canal or sinus
that are smaller than 50 points (4 cubic millimeters) are removed. The overlap scores use the
output unchanged.
"""
import argparse
import csv
import json
import math
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage
from scipy.spatial import cKDTree

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))
from src.curved_volume import DEPTH_IN_MM, DEPTH_STEP_MM, FRAME_BITE_ROW, FRAME_MIDDLE_COL, OUTSIDE, VOXEL_MM

CLASS_NAMES = ["background", "lower jaw", "upper jaw", "nerve canal", "sinus", "teeth"]
LABEL_COLORS = np.array([[0, 0, 0], [70, 110, 200], [60, 170, 170], [255, 40, 40], [255, 220, 0], [240, 240, 240]], np.uint8)
CANAL, SINUS = 3, 4
SPECK_POINTS = 50              # model pieces smaller than this are removed before measuring distances
NEAR_MM = 2.0                  # a true canal point counts as found if a model point is this close
TEETH_REGION_MM = 45.0         # thickness is measured this far to each side of the front of the jaw
THICKNESS_BELOW_BITE_MM = (12, 20)


def remove_specks(mask):
    """Drop connected pieces that are smaller than SPECK_POINTS."""
    pieces, n = ndimage.label(mask, structure=np.ones((3, 3, 3)))
    if n == 0:
        return mask
    sizes = np.bincount(pieces.ravel())
    keep = sizes >= SPECK_POINTS
    keep[0] = False
    return keep[pieces]


def middle_line(mask):
    """One point per column that holds the part: the average place of the part in that column.
    Returns [n, 3] in millimeters: (height, along the jaw curve, tongue-to-cheek)."""
    rows, cols, deps = np.nonzero(mask)
    if len(cols) == 0:
        return np.zeros((0, 3))
    n = np.bincount(cols, minlength=mask.shape[1])
    has = n > 0
    r = np.bincount(cols, weights=rows, minlength=mask.shape[1])[has] / n[has]
    d = np.bincount(cols, weights=deps, minlength=mask.shape[1])[has] / n[has]
    return np.stack([r * VOXEL_MM, np.flatnonzero(has) * VOXEL_MM, d * DEPTH_STEP_MM], axis=1)


def bottom_rows(mask):
    """For each column: the lowest row (largest row number) that holds the part, or -1 if none."""
    rows = np.arange(mask.shape[0])[:, None]
    return np.where(mask.any(axis=2), rows, -1).max(axis=0)


def measure_one(true_hu, true_lab, pred_hu, pred_lab):
    """All arrays are [rows, columns, depth]. Returns a dict of numbers for one scan."""
    nan = float("nan")
    res = {}
    inside = true_lab != OUTSIDE  # places outside the scan have no correct answer
    true_hu, pred_hu = true_hu.astype(np.float32), pred_hu.astype(np.float32)

    # 5. the numbers from the training script
    diff = (pred_hu - true_hu)[inside] / 4000.0
    res["err_hu"] = float(np.abs(diff).mean() * 4000)
    mean_square = float((diff ** 2).mean())
    res["psnr"] = float(-10 * math.log10(mean_square)) if mean_square > 0 else float("inf")
    for c in range(1, 6):
        t, p = true_lab == c, (pred_lab == c) & inside
        res[f"dice {CLASS_NAMES[c]}"] = float(2 * (t & p).sum() / (t.sum() + p.sum())) if t.any() else nan

    # 1. nerve canal
    true_line = middle_line(true_lab == CANAL)
    pred_line = middle_line(remove_specks((pred_lab == CANAL) & inside))
    res["canal mm"] = res["canal true to model mm"] = res["canal model to true mm"] = res["canal found share"] = nan
    if len(true_line):
        res["canal found share"] = 0.0
        if len(pred_line):
            d_true = cKDTree(pred_line).query(true_line)[0]
            d_pred = cKDTree(true_line).query(pred_line)[0]
            res["canal true to model mm"] = float(d_true.mean())
            res["canal model to true mm"] = float(d_pred.mean())
            res["canal mm"] = float((d_true.mean() + d_pred.mean()) / 2)
            res["canal found share"] = float((d_true <= NEAR_MM).mean())

    # 2. bottom edge of the sinus
    true_edge = bottom_rows(true_lab == SINUS)
    pred_edge = bottom_rows(remove_specks((pred_lab == SINUS) & inside))
    res["sinus edge mm"] = res["sinus edge signed mm"] = res["sinus columns found share"] = nan
    if (true_edge >= 0).any():
        both = (true_edge >= 0) & (pred_edge >= 0)
        res["sinus columns found share"] = float(both.sum() / (true_edge >= 0).sum())
        if both.any():
            d = (pred_edge[both] - true_edge[both]) * VOXEL_MM  # rows count downward, so positive = lower
            res["sinus edge mm"] = float(np.abs(d).mean())
            res["sinus edge signed mm"] = float(d.mean())

    # 3. thickness of the lower jaw, tongue side to cheek side
    jaw_true = np.isin(true_lab, (1, 3, 5))
    jaw_pred = np.isin(pred_lab, (1, 3, 5)) & inside
    half = int(round(TEETH_REGION_MM / VOXEL_MM))
    cols = slice(FRAME_MIDDLE_COL - half, FRAME_MIDDLE_COL + half + 1)
    for mm in THICKNESS_BELOW_BITE_MM:
        row = FRAME_BITE_ROW + int(round(mm / VOXEL_MM))
        t = jaw_true[row, cols].sum(axis=1) * DEPTH_STEP_MM
        p = jaw_pred[row, cols].sum(axis=1) * DEPTH_STEP_MM
        ok = t > 0
        res[f"jaw thickness true mm {mm}"] = float(t[ok].mean()) if ok.any() else nan
        res[f"jaw thickness model mm {mm}"] = float(p[ok].mean()) if ok.any() else nan
        res[f"jaw thickness diff mm {mm}"] = float(np.abs(p[ok] - t[ok]).mean()) if ok.any() else nan

    # 4. bone density
    for c in (1, 2):
        bone = true_lab == c
        res[f"{CLASS_NAMES[c]} HU true"] = float(true_hu[bone].mean()) if bone.any() else nan
        res[f"{CLASS_NAMES[c]} HU model"] = float(pred_hu[bone].mean()) if bone.any() else nan
    bone = (true_lab == 1) | (true_lab == 2)
    res["bone err HU"] = float(np.abs(pred_hu[bone] - true_hu[bone]).mean()) if bone.any() else nan
    return res


def colors(lab):
    return LABEL_COLORS[np.minimum(lab, 5)] * (lab[..., None] != OUTSIDE)


def save_overview(path, title, xray, pred_hu, pred_lab, true_hu, true_lab):
    """Model output next to the correct answer: one cut along the jaw curve and one cut seen from above."""
    outside = true_lab == OUTSIDE  # blank the model output where there is no correct answer
    pred_hu, pred_lab = np.where(outside, -1000, pred_hu), np.where(outside, OUTSIDE, pred_lab)
    at_curve = int(round(DEPTH_IN_MM / DEPTH_STEP_MM))  # the depth step that is on the jaw curve
    row = FRAME_BITE_ROW + int(12 / VOXEL_MM)            # 12 mm below where the teeth meet
    fig, ax = plt.subplots(5, 2, figsize=(18, 17), gridspec_kw=dict(height_ratios=[3, 3, 3, 1.2, 1.2]))
    ax[0, 0].imshow(xray, cmap="gray", vmin=0, vmax=1); ax[0, 0].set_title("input: X-ray")
    legend = "colors: blue lower jaw, teal upper jaw, red nerve canal, yellow sinus, white teeth"
    ax[0, 1].text(0.0, 0.5, f"{title}\n\n{legend}", fontsize=10, va="center", family="monospace")
    for col, (hu, lab, name) in enumerate([(pred_hu, pred_lab, "MODEL OUTPUT"), (true_hu, true_lab, "CORRECT ANSWER")]):
        ax[1, col].imshow(hu[:, :, at_curve], cmap="gray", vmin=-1000, vmax=2500); ax[1, col].set_title(f"{name}: cut along the jaw curve")
        ax[2, col].imshow(colors(lab[:, :, at_curve])); ax[2, col].set_title(f"{name}: parts on the same cut")
        ax[3, col].imshow(hu[row].T, cmap="gray", vmin=-1000, vmax=2500, aspect=DEPTH_STEP_MM / VOXEL_MM); ax[3, col].set_title(f"{name}: seen from above, 12 mm below the bite (top edge = tongue side)")
        ax[4, col].imshow(colors(lab[row]).transpose(1, 0, 2), aspect=DEPTH_STEP_MM / VOXEL_MM)
    for a in ax.ravel():
        a.axis("off")
    plt.subplots_adjust(left=0.01, right=0.99, top=0.97, bottom=0.01, wspace=0.02, hspace=0.12)
    plt.savefig(path, dpi=70); plt.close(fig)


def save_cross_sections(path, title, pred_hu, pred_lab, true_hu, true_lab, places_mm=(-35, -25, -15, 15, 25, 35)):
    """Cuts across the jaw (the view a dentist uses to plan an implant) at fixed distances from the
    front of the jaw. Top row: model output. Bottom row: correct answer.
    Red line = the model's nerve canal. Green line = the true nerve canal. Yellow = sinus."""
    outside = true_lab == OUTSIDE
    pred_hu, pred_lab = np.where(outside, -1000, pred_hu), np.where(outside, 0, pred_lab)
    true_lab = np.where(outside, 0, true_lab)
    top, bottom = FRAME_BITE_ROW - int(24 / VOXEL_MM), FRAME_BITE_ROW + int(56 / VOXEL_MM)
    fig, ax = plt.subplots(2, len(places_mm), figsize=(2.6 * len(places_mm), 11))
    for j, mm in enumerate(places_mm):
        col = FRAME_MIDDLE_COL + int(round(mm / VOXEL_MM))
        side = "patient's right" if mm < 0 else "patient's left"
        for i, (hu, name) in enumerate([(pred_hu, "MODEL"), (true_hu, "CORRECT")]):
            a = ax[i, j]
            a.imshow(hu[top:bottom, col, :], cmap="gray", vmin=-1000, vmax=2500, aspect=VOXEL_MM / DEPTH_STEP_MM)
            own = pred_lab if i == 0 else true_lab
            if (own[top:bottom, col, :] == SINUS).any():
                a.contour(own[top:bottom, col, :] == SINUS, levels=[0.5], colors="yellow", linewidths=1.2)
            if (true_lab[top:bottom, col, :] == CANAL).any():
                a.contour(true_lab[top:bottom, col, :] == CANAL, levels=[0.5], colors="lime", linewidths=1.5)
            if i == 0 and (pred_lab[top:bottom, col, :] == CANAL).any():
                a.contour(pred_lab[top:bottom, col, :] == CANAL, levels=[0.5], colors="red", linewidths=1.5)
            a.set_title(f"{name}\n{abs(mm)} mm from the front,\n{side}", fontsize=9)
            a.axis("off")
    fig.suptitle(f"{title}\ncuts across the jaw (left edge = tongue side, right edge = cheek side). "
                 "Red = model's nerve canal, green = true nerve canal, yellow = sinus", fontsize=10)
    plt.tight_layout(); plt.savefig(path, dpi=90); plt.close(fig)


def describe(res):
    """A few lines of text for the picture and the printed log."""
    f = lambda key, digits=1: "none" if math.isnan(res[key]) else f"{res[key]:.{digits}f}"
    return "\n".join([
        f"nerve canal: middle line off by {f('canal mm')} mm, share found {f('canal found share', 2)}",
        f"sinus bottom edge: off by {f('sinus edge mm')} mm",
        f"lower jaw thickness 12 mm below bite: model {f('jaw thickness model mm 12')} mm, true {f('jaw thickness true mm 12')} mm",
        f"lower jaw thickness 20 mm below bite: model {f('jaw thickness model mm 20')} mm, true {f('jaw thickness true mm 20')} mm",
        f"lower jaw bone brightness: model {f('lower jaw HU model', 0)}, true {f('lower jaw HU true', 0)} Hounsfield units",
        f"upper jaw bone brightness: model {f('upper jaw HU model', 0)}, true {f('upper jaw HU true', 0)} Hounsfield units",
        "overlap (0 to 1): " + ", ".join(f"{CLASS_NAMES[c]} {f('dice ' + CLASS_NAMES[c], 2)}" for c in range(1, 6)),
    ])


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True, help="folder made by prepare_tf3.py")
    ap.add_argument("--pred", required=True, help="folder made by predict.py")
    ap.add_argument("--out", required=True, help="folder for the numbers and pictures")
    ap.add_argument("--no-pictures", action="store_true")
    args = ap.parse_args()
    out = Path(args.out); out.mkdir(parents=True, exist_ok=True)
    numbers = sorted(p.stem for p in Path(args.pred).glob("*.npz"))
    if not numbers:
        raise SystemExit(f"ERROR: no model outputs found in {args.pred}")

    rows = []
    for num in numbers:
        with np.load(Path(args.data) / f"{num}.npz") as f:
            xray, true_hu, true_lab = f["xray"].astype(np.float32), f["hu"], f["labels"]
        with np.load(Path(args.pred) / f"{num}.npz") as f:
            pred_hu, pred_lab = f["hu"], f["labels"]
        if pred_hu.shape != true_hu.shape:
            raise SystemExit(f"ERROR: scan {num}: model output {pred_hu.shape} and correct answer {true_hu.shape} have different sizes")
        res = measure_one(true_hu, true_lab, pred_hu, pred_lab)
        rows.append(dict(scan=num, **res))
        text = describe(res)
        print(f"scan {num}\n  " + text.replace("\n", "\n  "), flush=True)
        if not args.no_pictures:
            save_overview(out / f"overview_{num}.png", f"scan {num}\n\n{text}", xray, pred_hu, pred_lab, true_hu, true_lab)
            save_cross_sections(out / f"cross_{num}.png", f"scan {num}", pred_hu, pred_lab, true_hu, true_lab)

    keys = [k for k in rows[0] if k != "scan"]
    table = np.array([[r[k] for k in keys] for r in rows], dtype=float)
    counted = (~np.isnan(table)).sum(axis=0)
    with np.errstate(invalid="ignore"):
        mean = np.where(counted > 0, np.nansum(table, axis=0) / np.maximum(counted, 1), np.nan)
        spread = np.sqrt(np.where(counted > 0, np.nansum((table - mean) ** 2, axis=0) / np.maximum(counted, 1), np.nan))
    summary = {k: dict(average=float(mean[i]), spread=float(spread[i]), scans_counted=int(counted[i])) for i, k in enumerate(keys)}
    # Does the model tell a scan with dense bone from a scan with less dense bone?
    # 1 = the model's order of the scans is the same as the true order, 0 = no link.
    for name in ("lower jaw", "upper jaw"):
        t, p = table[:, keys.index(f"{name} HU true")], table[:, keys.index(f"{name} HU model")]
        ok = ~np.isnan(t) & ~np.isnan(p)
        summary[f"{name} HU link between model and true"] = float(np.corrcoef(t[ok], p[ok])[0, 1]) if ok.sum() >= 3 else float("nan")

    with open(out / "results.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["scan"] + keys)
        w.writeheader()
        w.writerows(rows)
        w.writerow(dict(scan="average", **{k: float(mean[i]) for i, k in enumerate(keys)}))
    (out / "summary.json").write_text(json.dumps(dict(scans=numbers, numbers=summary), indent=1))

    print(f"\nAVERAGE OVER {len(numbers)} SCANS (spread = how much the scans differ from the average)")
    for i, k in enumerate(keys):
        print(f"  {k:32s} {mean[i]:9.2f}   spread {spread[i]:7.2f}   scans counted {counted[i]}")
    for name in ("lower jaw", "upper jaw"):
        print(f"  {name} HU link between model and true: {summary[f'{name} HU link between model and true']:.2f}")
    print(f"numbers and pictures are in {out}")


if __name__ == "__main__":
    main()


The next two cells run the model on the 5 check scans and compare the scores with the scores from training.

In [ ]:
!python /kaggle/working/project/scripts/predict.py --data /tmp/prepared --model "$MODEL" --out /tmp/predictions/check --group check
!python /kaggle/working/project/scripts/measure.py --data /tmp/prepared --pred /tmp/predictions/check --out /kaggle/working/results/check --no-pictures

In [ ]:
import json, pandas as pd
log = pd.read_csv(os.path.dirname(MODEL) + "/log.csv")
epoch = json.load(open("/tmp/predictions/check/about.json"))["epoch"]
during_training = log[log.epoch == epoch].iloc[0]
now = pd.read_csv("/kaggle/working/results/check/results.csv").iloc[-1]
names = ["err_hu", "psnr", "dice lower jaw", "dice upper jaw", "dice nerve canal", "dice sinus", "dice teeth"]
table = pd.DataFrame({"during training": [during_training[n] for n in names], "now": [float(now[n]) for n in names]}, index=names)
display(table.round(3))
worst = (table["during training"] - table["now"]).abs().div(table["during training"].abs()).max()
print("SAME AS DURING TRAINING" if worst < 0.01 else "DIFFERENT FROM TRAINING: the model file may not be loaded correctly")

The next cell runs the model on the 10 test scans and measures the result in millimeters.

In [ ]:
!python /kaggle/working/project/scripts/predict.py --data /tmp/prepared --model "$MODEL" --out /kaggle/working/predictions/test --group test
!python /kaggle/working/project/scripts/measure.py --data /tmp/prepared --pred /kaggle/working/predictions/test --out /kaggle/working/results/test

In [ ]:
from IPython.display import Image, display
display(pd.read_csv("/kaggle/working/results/test/results.csv").round(2).T)
first = sorted(glob.glob("/kaggle/working/results/test/overview_*.png"))[0]
display(Image(first, width=1100))
display(Image(first.replace("overview_", "cross_"), width=1100))